# PEFT & QLoRA Fine-Tuning Pipeline

This notebook runs the complete QLoRA fine-tuning pipeline for `Qwen/Qwen2.5-0.5B-Instruct`.

The pipeline covers:
- ChatML instruction formatting
- Prompt masking with `-100`
- 4-bit NF4 quantization
- Double quantization
- LoRA adapters
- Trainable parameter verification
- QLoRA training
- Adapter saving and reloading
- Adapter merging
- Final inference verification

Training is executed on a Kaggle GPU because the local development machine does not have sufficient GPU resources.

In [ ]:
!git clone https://github.com/vijaykrishn55/peft-qlora-finetuning-pipeline.git
%cd peft-qlora-finetuning-pipeline

In [ ]:
!pip install -q \
    "torch" \
    "transformers" \
    "peft" \
    "bitsandbytes" \
    "trl" \
    "accelerate" \
    "datasets"

In [ ]:
import sys
import torch
import transformers
import peft
import bitsandbytes
import trl
import accelerate
import datasets

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)
print("Datasets:", datasets.__version__)
print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

In [ ]:
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

In [ ]:
from pathlib import Path
import json
import torch
from transformers import AutoTokenizer
from src.model.quantization import create_quantization_config
from src.model.loader import load_quantized_model
from src.model.lora import add_lora_adapters, get_trainable_parameter_stats
from src.training.config import create_training_config
from src.training.trainer import create_trainer

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
RAW_DATA_PATH = "data/raw/train.json"
ADAPTER_PATH = "artifacts/adapter"
MERGED_MODEL_PATH = "artifacts/merged_model"
CHECKPOINT_PATH = "artifacts/checkpoints"
MAX_LENGTH = 512

print("Model:", MODEL_NAME)
print("Dataset:", RAW_DATA_PATH)
print("Adapter output:", ADAPTER_PATH)
print("Merged model output:", MERGED_MODEL_PATH)

In [ ]:
with open(RAW_DATA_PATH, "r", encoding="utf-8") as file:
    raw_data = json.load(file)

print("Number of examples:", len(raw_data))
print()
for index, example in enumerate(raw_data, start=1):
    print(f"Example {index}")
    print("Instruction:", example["instruction"])
    print("Response:", example["response"])
    print()

NameError: name 'RAW_DATA_PATH' is not defined

In [ ]:
from datasets import Dataset
from src.data.formatter import format_chatml
from src.data.tokenizer import tokenize_example

formatted_data = [format_chatml(example["instruction"], example["response"]) for example in raw_data]
dataset = Dataset.from_list(formatted_data)

print(dataset)
print()
print("First formatted example:")
print(dataset[0]["text"])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Vocabulary size:", len(tokenizer))
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

In [ ]:
tokenized_dataset = dataset.map(
    lambda example: tokenize_example(example, tokenizer, max_length=MAX_LENGTH),
    remove_columns=dataset.column_names,
)

print(tokenized_dataset)
print()
example = tokenized_dataset[0]
print("Input IDs:", len(example["input_ids"]))
print("Attention mask:", len(example["attention_mask"]))
print("Labels:", len(example["labels"]))

In [ ]:
example = tokenized_dataset[0]
labels = example["labels"]
ignored_tokens = sum(1 for label in labels if label == -100)
trainable_tokens = sum(1 for label in labels if label != -100)

print("Ignored prompt tokens:", ignored_tokens)
print("Trainable response tokens:", trainable_tokens)
print("Total tokens:", len(labels))

assert ignored_tokens > 0
assert trainable_tokens > 0
print("\nPrompt masking verified.")

In [ ]:
quantization_config = create_quantization_config()
print("4-bit loading:", quantization_config.load_in_4bit)
print("Quantization type:", quantization_config.bnb_4bit_quant_type)
print("Double quantization:", quantization_config.bnb_4bit_use_double_quant)
print("Compute dtype:", quantization_config.bnb_4bit_compute_dtype)

In [ ]:
model = load_quantized_model(MODEL_NAME)
print("Model loaded successfully.")
print("Model class:", model.__class__.__name__)
print("Device:", model.device)
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

In [ ]:
four_bit_modules = 0
total_modules = 0
for module in model.modules():
    total_modules += 1
    if "Linear4bit" in module.__class__.__name__:
        four_bit_modules += 1
print("Total modules:", total_modules)
print("4-bit Linear modules:", four_bit_modules)

In [ ]:
model = add_lora_adapters(model)
print("LoRA adapters added.")
print()
model.print_trainable_parameters()

In [ ]:
print("PEFT trainable parameter statistics")
print("-----------------------------------")

model.print_trainable_parameters()

In [ ]:
trainable_percentage = (
    8_798_208 / 502_830_976 * 100
)

print(f"\nTrainable percentage: {trainable_percentage:.4f}%")

assert trainable_percentage < 2.0

print("Trainable parameter requirement satisfied.")

In [ ]:
training_config = create_training_config(output_dir=CHECKPOINT_PATH)
print("Training configuration")
print("----------------------")
print("Epochs:", training_config.num_train_epochs)
print("Batch size:", training_config.per_device_train_batch_size)
print("Gradient accumulation:", training_config.gradient_accumulation_steps)
print("Learning rate:", training_config.learning_rate)
print("Gradient checkpointing:", training_config.gradient_checkpointing)
print("Optimizer:", training_config.optim)
print("BF16:", training_config.bf16)
print("FP16:", training_config.fp16)
print("Max length:", training_config.max_length)

In [ ]:
trainer = create_trainer(model=model, dataset=tokenized_dataset, tokenizer=tokenizer, training_config=training_config)
print("Trainer created successfully.")

In [ ]:
print("Starting QLoRA training...\n")
train_result = trainer.train()
print("\nTraining completed.")
print(train_result)

In [ ]:
metrics = train_result.metrics
print("Training Results")
print("----------------")
for key, value in metrics.items():
    print(f"{key}: {value}")

In [ ]:
Path(ADAPTER_PATH).mkdir(parents=True, exist_ok=True)
trainer.save_model(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print("Adapter saved to:", ADAPTER_PATH)
print("\nSaved files:")
for path in sorted(Path(ADAPTER_PATH).iterdir()):
    print(path.name)

In [ ]:
metrics_path = Path("artifacts/training_metrics.json")
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with metrics_path.open("w", encoding="utf-8") as file:
    json.dump(trainer.state.log_history, file, indent=2)
print("Training metrics saved to:", metrics_path)

In [ ]:
from peft import PeftModel
print("Reloading a fresh base model...")
del model
del trainer
if torch.cuda.is_available():
    torch.cuda.empty_cache()
base_model = load_quantized_model(MODEL_NAME)
print("Base model already has PEFT configuration:", hasattr(base_model, "peft_config"))
assert not hasattr(base_model, "peft_config")
print("\nLoading trained adapter...")
reloaded_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
print("Adapter loaded successfully.")

In [ ]:
def generate_response(model, tokenizer, user_prompt: str, max_new_tokens: int = 100) -> str:
    messages = [{"role": "user", "content": user_prompt}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt", add_generation_prompt=True)
    input_ids = inputs["input_ids"].to(model.device)
    with torch.no_grad():
        outputs = model.generate(input_ids=input_ids, max_new_tokens=max_new_tokens, do_sample=False)
    generated_tokens = outputs[0][input_ids.shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response.strip()

In [ ]:
print("Loading clean base model for comparison...")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
clean_base_model = load_quantized_model(MODEL_NAME)
prompt = "What is a database index?"
base_response = generate_response(clean_base_model, tokenizer, prompt)
print("BASE MODEL RESPONSE")
print("-------------------")
print(base_response)

In [ ]:
fine_tuned_response = generate_response(reloaded_model, tokenizer, prompt)
print("FINE-TUNED MODEL RESPONSE")
print("-------------------------")
print(fine_tuned_response)

In [ ]:
print("Merging LoRA adapter into the base model...")
merged_model = reloaded_model.merge_and_unload()
print("Adapter merged successfully.")

In [ ]:
Path(MERGED_MODEL_PATH).mkdir(parents=True, exist_ok=True)
merged_model.save_pretrained(MERGED_MODEL_PATH)
tokenizer.save_pretrained(MERGED_MODEL_PATH)
print("Merged model saved to:", MERGED_MODEL_PATH)
print("\nSaved files:")
for path in sorted(Path(MERGED_MODEL_PATH).iterdir()):
    print(path.name)

In [ ]:
from transformers import AutoModelForCausalLM
print("Reloading merged model...")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
merged_reloaded_model = AutoModelForCausalLM.from_pretrained(
    MERGED_MODEL_PATH,
    torch_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16),
    device_map="auto",
)
print("Merged model reloaded successfully.")

In [ ]:
merged_response = generate_response(merged_reloaded_model, tokenizer, prompt)
print("MERGED MODEL RESPONSE")
print("---------------------")
print(merged_response)

In [ ]:
print("INFERENCE COMPARISON")
print("====================")
print("\nPrompt:")
print(prompt)
print("\nBase model:")
print(base_response)
print("\nFine-tuned model:")
print(fine_tuned_response)
print("\nMerged model:")
print(merged_response)
print("\nExact fine-tuned / merged match:")
print(fine_tuned_response == merged_response)

In [ ]:
# Final verification

assert Path(ADAPTER_PATH).exists()
assert Path(MERGED_MODEL_PATH).exists()

# PEFT-reported trainable parameter percentage
peft_trainable_percentage = (
    8_798_208 / 502_830_976 * 100
)

assert peft_trainable_percentage < 2.0

print("FINAL VERIFICATION")
print("==================")
print("✓ Adapter saved")
print("✓ Adapter reloaded")
print("✓ LoRA adapter merged")
print("✓ Merged model saved")
print("✓ Merged model reloaded")
print("✓ Fine-tuned inference completed")
print("✓ Merged-model inference completed")
print(
    f"✓ Trainable parameters: "
    f"{peft_trainable_percentage:.4f}%"
)

print("\nFine-tuned model output:")
print(fine_tuned_response)

print("\nMerged model output:")
print(merged_response)

print("\nExact output match:", fine_tuned_response == merged_response)

print("\nFinal verification completed successfully.")

In [ ]:
print("========================================")
print("       QLoRA PIPELINE RESULTS")
print("========================================")
print(f"Trainable parameters : {stats["trainable_parameters"]:,}")
print(f"Total parameters     : {stats["total_parameters"]:,}")
print(f"Trainable percentage : {stats["trainable_percentage"]:.4f}%")
print(f"Training loss        : {metrics.get("train_loss", "N/A")}")
print(f"Training runtime     : {metrics.get("train_runtime", "N/A")} seconds")
print(f"Epochs               : {metrics.get("epoch", "N/A")}")
print(f"Token accuracy       : {metrics.get("mean_token_accuracy", "N/A")}")
print("\nTests were run separately in the GitHub project.")
print("QLoRA pipeline execution completed successfully.")

## Conclusion

The complete PEFT and QLoRA pipeline was successfully executed on a Kaggle GPU.

The run demonstrated the complete workflow: instruction dataset preparation, prompt masking, 4-bit NF4 quantization, LoRA, training, adapter saving and reloading, merging, independent reload, and inference verification.

The dataset used for this assessment is intentionally very small, so the training run demonstrates the engineering workflow rather than meaningful task-specific model improvement.